# Accuracy results

Golden-set accuracy for each candidate model, from the runs written by `run_accuracy.py` (latest complete 175-ticket run per model in `accuracy_results/`).

- Every golden ticket was posted to the triage service (`POST /tickets`) with the frozen prompt, settings and parser.
- Denominator is always all 175 golden tickets. Unparseable replies (`INVALID`) and failed requests (`ERROR`) count as wrong and are shown separately.
- Requirements from `performance-requirements.md`: **R5** overall accuracy ≥ 80%, **R6** every category ≥ 65%.

In [13]:
import importlib
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "accuracy.py").is_file() else Path.cwd() / "evaluation"
sys.path.insert(0, str(HERE))

import accuracy, classifier
importlib.reload(accuracy)

SGT = timezone(timedelta(hours=8))

runs = accuracy.load_runs()
for model, (summary, _) in runs.items():
    start = datetime.fromisoformat(summary["started_utc"]).astimezone(SGT)
    end = datetime.fromisoformat(summary["finished_utc"]).astimezone(SGT)
    minutes = round((end - start).total_seconds() / 60)
    problems = summary["reconciliation_problems"]
    print(f"{model}")
    print(f"  Run ID:         {summary['run_id']}")
    print(f"  Time (SGT):     {start:%a %d %b %Y, %H:%M} to {end:%H:%M} ({minutes} min)")
    print(f"  Git commit:     {summary['git_head']}")
    print(f"  Reconciliation: {'clean' if not problems else problems}\n")
missing = [m for m in classifier.MODELS if m not in runs]
print("Missing models:", ", ".join(missing) if missing else "none")

llama3.2:1b
  Run ID:         acc_llama3.2_1b_20261005T160744Z
  Time (SGT):     Tue 06 Oct 2026, 00:07 to 00:15 (8 min)
  Git commit:     df879e6
  Reconciliation: clean

qwen2.5:1.5b
  Run ID:         acc_qwen2.5_1.5b_20261005T161551Z
  Time (SGT):     Tue 06 Oct 2026, 00:15 to 00:22 (7 min)
  Git commit:     df879e6
  Reconciliation: clean

phi3.5:3.8b
  Run ID:         acc_phi3.5_3.8b_20261005T162247Z
  Time (SGT):     Tue 06 Oct 2026, 00:22 to 00:37 (15 min)
  Git commit:     df879e6
  Reconciliation: clean

qwen2.5:7b
  Run ID:         acc_qwen2.5_7b_20261005T163749Z
  Time (SGT):     Tue 06 Oct 2026, 00:37 to 01:02 (25 min)
  Git commit:     df879e6
  Reconciliation: clean

Missing models: none


## 1. Overall accuracy and requirement checks

In [2]:
accuracy.print_summary(runs)

model          correct    acc        95% CI INVALID ERROR R5>=80% R6>=65%  worst category
llama3.2:1b     38/175  21.7%  16.2%-28.4%        2     0    FAIL    FAIL  Mortgage (0%)
qwen2.5:1.5b    92/175  52.6%  45.2%-59.8%        0     0    FAIL    FAIL  Bank account or service (24%)
phi3.5:3.8b    133/175  76.0%  69.2%-81.7%        0     0    FAIL    FAIL  Credit card (62%)
qwen2.5:7b     135/175  77.1%  70.4%-82.7%        0     0    FAIL    FAIL  Money transfer or service (56%)


## 2. Per-category accuracy (correct / total)

In [3]:
accuracy.print_per_category(runs)

gold category                n    llama3.2:1b   qwen2.5:1.5b    phi3.5:3.8b     qwen2.5:7b
Credit reporting            36  32/36    89%   31/36    86%   27/36    75%   33/36    92% 
Debt collection             24   3/24    12%*  10/24    42%*  18/24    75%   19/24    79% 
Mortgage                    22   0/22     0%*  10/22    45%*  18/22    82%   16/22    73% 
Credit card                 21   0/21     0%*  12/21    57%*  13/21    62%*  16/21    76% 
Bank account or service     33   3/33     9%*   8/33    24%*  27/33    82%   25/33    76% 
Consumer loan               23   0/23     0%*  15/23    65%   18/23    78%   17/23    74% 
Money transfer or service   16   0/16     0%*   6/16    38%*  12/16    75%    9/16    56%*
* below the 65% per-category requirement (R6)


## 3. Confusion matrices

In [4]:
for model, (_, records) in runs.items():
    accuracy.print_confusion(model, records)
    print()

llama3.2:1b: rows = gold label, columns = prediction
            CredRep     Debt    Mortg     Card     Bank     Loan Transfer  INVALID  total
CredRep          32        4        0        0        0        0        0        0     36
Debt             20        3        0        0        0        0        0        1     24
Mortg            16        5        0        0        0        0        0        1     22
Card             17        4        0        0        0        0        0        0     21
Bank             24        6        0        0        3        0        0        0     33
Loan             15        8        0        0        0        0        0        0     23
Transfer          9        7        0        0        0        0        0        0     16

qwen2.5:1.5b: rows = gold label, columns = prediction
            CredRep     Debt    Mortg     Card     Bank     Loan Transfer  total
CredRep          31        2        1        1        0        1        0     36
Debt      

## 4. Most frequent confusions (gold → predicted)

In [5]:
accuracy.print_top_confusions(runs)

llama3.2:1b   Bank account or service -> Credit reporting (24); Debt collection -> Credit reporting (20); Credit card -> Credit reporting (17)
qwen2.5:1.5b  Bank account or service -> Credit card (16); Debt collection -> Credit card (6); Mortgage -> Consumer loan (5)
phi3.5:3.8b   Credit reporting -> Debt collection (5); Credit card -> Bank account or service (5); Mortgage -> Consumer loan (4)
qwen2.5:7b    Money transfer or service -> Bank account or service (5); Credit reporting -> Debt collection (3); Debt collection -> Credit reporting (3)


## 5. Predictions vs results

Frozen predictions from `prediction-record.md` against the measured accuracy and the warm single-request latency of the same run (service `total_ms` from the request log, one request at a time).

In [6]:
accuracy.print_predictions(runs)

model         pred acc  actual   diff pred lat  median    p95
llama3.2:1b        68%   21.7% -46.3pp     2.5s   2.54s  4.74s
qwen2.5:1.5b       73%   52.6% -20.4pp     3.5s   1.68s  3.62s
phi3.5:3.8b        79%   76.0%  -3.0pp     7.0s   4.53s  9.50s
qwen2.5:7b         84%   77.1%  -6.9pp    12.0s   7.57s 15.63s
Latency = service total_ms from the request log for this run (sequential, warm, one request at a time)


## 6. Export Results

In [7]:
accuracy.export(runs)

Wrote accuracy_summary.csv, accuracy_per_category.csv and confusion_<model>.csv to accuracy_results
